# 2 — Data Preprocessing Pipeline

**Project:** Predictive Modeling of US Used Vehicle Prices  
**Course:** ENGR422 — Applied Machine Learning  
**Authors:** Eren Acar Başaran, Ahmet Aybars Pektaş  

---

This notebook covers **Work Package 2 — Preprocessing Pipeline** from the project proposal.

**Deliverables:**
- D2.1: Cleaned and Split Dataset ready for modeling
- D2.2: Scikit-Learn Preprocessing Pipeline Script

## 2.1 — Imports & Load Raw Data

Import libraries (Pandas, NumPy, Scikit-Learn). Load the raw dataset from `../data/vehicles.csv` and print the shape. No filtering here — that happens in section 2.3.

In [3]:
# 2.1 — Imports & Load Raw Data
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
DATA_PATH = "../data/vehicles.csv"

# Load raw dataset
df = pd.read_csv(DATA_PATH)
print(f"Raw dataset shape: {df.shape}")

Raw dataset shape: (426880, 26)


## 2.2 — Feature Selection & Column Dropping

Based on the EDA findings, drop columns that are not useful for modeling:
- **Identifiers:** `id`, `url`, `region_url`, `image_url`, `VIN` — not predictive
- **Free-text:** `description` — NLP out of scope
- **Geography:** `lat`, `long`, `region` — `state` kept instead
- **Excessive missing:** `county` (100%), `size` (71.8%), `cylinders` (41.6%) — too many NaN to be useful
- **Temporal:** `posting_date` — not relevant to price

Print the remaining columns and shape after dropping.

In [4]:
# 2.2 — Feature Selection & Column Dropping

# Identifiers — no predictive value
# Free-text / URL fields — not used (text analysis is out of scope)
# Columns with excessive missing data or no data at all (county is 100% NaN)
# Location coordinates — too granular, state is kept instead
# posting_date — not a vehicle feature
# region — redundant with state
# size — over 70% missing in EDA
cols_to_drop = [
    "id", "url", "region_url", "image_url",  # identifiers / URLs
    "description",                             # free-text, out of scope
    "VIN",                                     # identifier
    "county",                                  # 100% NaN
    "lat", "long",                             # coordinates, too granular
    "posting_date",                            # not a vehicle feature
    "region",                                  # redundant with state
    "size",                                    # >70% missing
    "cylinders",                               # 41.6% missing, correlated with manufacturer/model
]

df = df.drop(columns=cols_to_drop, errors="ignore")

# state kept for potential use in future iterations
print(f"Shape after dropping columns: {df.shape}")
print(f"Remaining columns: {df.columns.tolist()}")

Shape after dropping columns: (426880, 13)
Remaining columns: ['price', 'year', 'manufacturer', 'model', 'condition', 'fuel', 'odometer', 'title_status', 'transmission', 'drive', 'type', 'paint_color', 'state']


## 2.3 — Outlier Filtering

Apply hard-bound filtering based on EDA findings:
- Drop rows where `price` is NaN.
- Keep rows where `price` >= $500 **and** `price` <= $100,000 (removes spam listings and unrealistic values).
- Drop rows where `odometer` > 300,000 miles.

Report the number of rows before and after filtering.

In [5]:
# 2.3 — Outlier Filtering
rows_before = len(df)

# Drop rows where price is NaN
df = df.dropna(subset=["price"])

# Price filter: hard min $500, hard max $100,000
# Keeps legitimate vehicles while removing $0/$1 spam and unrealistic listings
df = df[(df["price"] >= 500) & (df["price"] <= 100_000)]

# Odometer: drop rows above 300,000 miles
df = df[df["odometer"].fillna(0) <= 300_000]

rows_after = len(df)
print(f"Before filtering: {rows_before:,} rows")
print(f"After filtering:  {rows_after:,} rows")
print(f"Rows removed:     {rows_before - rows_after:,}")

Before filtering: 426,880 rows
After filtering:  381,427 rows
Rows removed:     45,453


## 2.4 — Train-Test Split

**!! CRITICAL: Split BEFORE any imputation or encoding to prevent data leakage !!**

- Separate features (`X`) and target (`y = price`).
- Perform an 80/20 train-test split using `train_test_split` with a fixed `random_state` for reproducibility.
- Report the shape of train and test sets.

In [ ]:
# 2.4 — Train-Test Split
# !! CRITICAL: Split BEFORE any imputation or encoding to prevent data leakage !!

X = df.drop(columns=["price"])
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

## 2.5 — Missing Value Imputation

Build imputation strategies (fit strictly on training data only):
- **Numerical features** (`year`, `odometer`): Impute with median values.
- **Categorical features** (`manufacturer`, `condition`, `fuel`, `transmission`, `drive`, `type`, etc.): Impute with the mode (most frequent value).

Use Scikit-Learn's `SimpleImputer` within the pipeline so that imputation statistics are learned from training data and applied consistently to test data.

## 2.6 — Feature Encoding

Encode categorical features for model consumption:
- **Low-cardinality features** (e.g., `fuel`, `transmission`, `condition`, `drive`, `type`, `title_status`): Use One-Hot Encoding or Ordinal Encoding as appropriate.
- **High-cardinality features** (e.g., `manufacturer`, `model`, `state`): Apply **Target Encoding** to avoid dimensionality explosion.

Wrap all encoding in Scikit-Learn `ColumnTransformer` to keep the pipeline clean.

## 2.7 — Feature Scaling (Optional)

- Apply `StandardScaler` to numerical features for the Linear Regression model (tree-based models do not require scaling, but it won't hurt).
- This step can be toggled on/off per model within the final pipeline.

## 2.8 — Assemble the Scikit-Learn Pipeline

Build the full `sklearn.pipeline.Pipeline` object that chains:
1. `ColumnTransformer` (imputation + encoding for numerical and categorical columns)
2. Optional scaler
3. Placeholder for the estimator (to be swapped per model notebook)

Save/export the preprocessing pipeline so the model notebooks can import it directly.

## 2.9 — Verification & Sanity Checks

- Transform the training data through the pipeline and verify the output shape.
- Check for any remaining NaN values.
- Print a sample of transformed feature names.
- Confirm no test data was used during fitting.